# Gerar o pacote final (zip) do app 3 — Mobilidade Urbana — v2
Gera `Mobilidade_v2.zip` (só pt-br) com o comando de download do ExB e copia para a pasta **Saidas** do projeto aberto no Pro.
Esse zip serve para a **prévia pública** (GitHub Pages) e para o **deploy oficial** pelo João.
Antes: o app 3 tem que estar **publicado** no construtor (Salvar → Publicar).

In [ ]:
# ===== CÉLULA 1 — Configuração =====
import os, json, datetime, subprocess, shutil, glob, time, zipfile, re
NB_NOME = "Gerar_Zip_Mobilidade_v2"
EXB = r"C:\ExB_1.19"
APP_ID = "3"
ZIP_NOME = "Mobilidade_v2.zip"
try:
    import arcpy
    PASTA_SAIDA = os.path.join(arcpy.mp.ArcGISProject("CURRENT").homeFolder, "Saidas")
except Exception:
    PASTA_SAIDA = os.path.join(os.path.expanduser("~"), "Documents", "Saidas_Notebooks")
os.makedirs(PASTA_SAIDA, exist_ok=True)
ARQ_TXT = os.path.join(PASTA_SAIDA, NB_NOME + ".txt")
with open(ARQ_TXT, "w", encoding="utf-8") as f:
    f.write(f"{NB_NOME} — executado em {datetime.datetime.now():%d/%m/%Y %H:%M:%S}\n\n")
def log(*a):
    s = " ".join(str(x) for x in a); print(s)
    with open(ARQ_TXT, "a", encoding="utf-8") as f: f.write(s + "\n")
log("▶ CÉLULA 1 — Configuração: iniciando...")
pub = os.path.join(EXB, "server", "public", "apps", APP_ID, "config.json")
with open(pub, encoding="utf-8") as f:
    c = json.load(f)
if c.get("__not_publish") or not c.get("pages"):
    raise RuntimeError("O app 3 ainda não está publicado. No construtor: Salvar → Publicar, e rode de novo.")
tem_servicos = "widget_servicos" in c.get("widgets", {})
log("  App 3 publicado — widgets:", len(c["widgets"]), "| visual DEVs (Serviços ao Passageiro):", "SIM" if tem_servicos else "NÃO — publique a v13 antes!")
log("  Pasta de saída:", PASTA_SAIDA)
log("✔ CÉLULA 1 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 2 — Gerar o zip =====
log("\n▶ CÉLULA 2 — Geração do zip: iniciando (pode levar 1–3 minutos)...")
inicio = time.time()
js = "require('./server/src/middlewares/dev/apps/app-download.js').zipApp('%s', '%s', '', {locales:['pt-br']});" % (APP_ID, ZIP_NOME)
r = subprocess.run(["node", "-e", js], cwd=EXB, capture_output=True, text=True, shell=True, timeout=900)
if r.stdout.strip(): log("  saída:", r.stdout.strip()[-500:])
if r.stderr.strip(): log("  avisos:", r.stderr.strip()[-500:])
achado, tam_ant = None, -1
for _ in range(180):
    cands = [p for p in glob.glob(os.path.join(EXB, "**", ZIP_NOME), recursive=True) if os.path.getmtime(p) >= inicio - 5]
    if cands:
        p = max(cands, key=os.path.getmtime); t = os.path.getsize(p)
        if t == tam_ant and t > 0: achado = p; break
        tam_ant = t
    time.sleep(2)
if not achado:
    raise RuntimeError("Não encontrei o zip gerado. Confira se o ExB está aberto e mande este .txt.")
destino = os.path.join(PASTA_SAIDA, ZIP_NOME)
shutil.copy2(achado, destino)
log(f"  Zip: {destino}  ({os.path.getsize(destino)/1e6:.1f} MB)")
log("✔ CÉLULA 2 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 3 — Conferência do pacote =====
log("\n▶ CÉLULA 3 — Conferência: iniciando...")
with zipfile.ZipFile(destino) as z:
    nomes = z.namelist()
    cfgs = [n for n in nomes if n.endswith("/config.json") and "/cdn/" in "/" + n and n.count("/") == 2]
    cfg_txt = z.read(cfgs[0]).decode("utf-8") if cfgs else ""
widgets = sorted({n.split("/")[3] for n in nomes if n.count("/") >= 4 and n.split("/")[2] == "widgets" and n.split("/")[3] in ("horarios-linha", "servicos-passageiro")})
log("  Arquivos no zip:", len(nomes), "| config:", cfgs[0] if cfgs else "NÃO ENCONTRADO")
log("  Widgets próprios no pacote:", widgets or "NENHUM")
sus = re.findall(r'"(clientId|token|password|senha)"\s*:\s*"([^"]+)"', cfg_txt, flags=re.I)
log("  Segredos no config:", "nenhum" if not sus else sus)
log("✔ CÉLULA 3 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 4 — Próximo passo =====
log("\n▶ CÉLULA 4 — Próximo passo: iniciando...")
log("  Avise no chat: vou pegar o Mobilidade_v2.zip desta pasta Saidas, conferir e publicar a prévia em /mobilidade.")
log("  Prévia: https://jrnegrao.github.io/cctv/mobilidade/?locale=pt-br")
log(f"  Saída completa: {ARQ_TXT}")
log("  Ao terminar: feche esta aba e remova o notebook do projeto (Remove, não Delete).")
log("✔ CÉLULA 4 concluída com sucesso.")
